# Adım 7 — Dashboard ve GörselleştirmeBu notebook, **Streamlit Dashboard** mimarisini, sayfalarını ve zorunlu görsellerin karşılanma durumunu belgeler.## Dashboard Teknoloji Stack'i| Bileşen | Teknoloji | Görevi ||---------|-----------|-------|| **Frontend** | Streamlit + Plotly | 6 sayfalık interaktif dashboard || **Backend API** | Flask + SSE | Kafka consumer + Delta Lake stats (canlı) || **Veri Kaynağı** | Delta Lake + MLflow SQLite | Gold tablo + model metrikleri || **Tema** | Custom dark theme | `theme.py` — glassmorphism kartlar, badge'ler |## Erişim| Servis | URL ||--------|-----|| Streamlit Dashboard | http://localhost:8501 || Dashboard API | http://localhost:5001 || MLflow UI | http://localhost:5000 |

## Dashboard Sayfaları (6 Adet)### 1. 🏠 Ana Sayfa (`app.py`)- Pipeline diyagramı (Kafka → Bronze → Silver → Gold → ML → MLflow)- Hızlı KPI'lar: Gold satır sayısı, kolon, türetilmiş feature, en iyi model### 2. 📊 Genel Bakış (`1_Genel_Bakış.py`)- Problem tanımı, mimari diyagram- Delta Lake katman istatistikleri (satır/kolon/boyut)- 5 model kartı (accuracy)- 5 türetilmiş özellik kartı### 3. 🌊 Veri Akışı (`2_Veri_Akışı.py`)- **SSE canlı izleme** — sayfa yenilemesiz- Kafka durum kartı (bağlantı + mesaj sayısı + rate)- Animasyonlu veri akış diyagramı (Kafka → Bronze → Silver → Gold)- Canvas time series — satır sayısı geçmişi (son 120s)- Sankey diyagramı — katmanlar arası veri hacmi### 4. 🔍 EDA (`3_EDA.py`)- Saldırı tipi dağılımı (horizontal bar + donut/pie chart)- Zaman serisi trendi (saatlik mesaj hacmi — area chart)- Eksik değer analizi (bar chart)- Sayısal feature dağılımları (histogram)- Korelasyon ısı haritası (heatmap)### 5. ⚙️ Feature Engineering (`4_Feature_Engineering.py`)- 5 özellik kartı (formül + iş mantığı + hedef saldırı)- Violin plot — her feature'ın saldırı tipine göre dağılımı- Üretim akış diyagramı (Silver → Gold)### 6. 🤖 Model Karşılaştırma (`5_Model_Karşılaştırma.py`)- Performans kartları (accuracy sıralaması — 🥇🥈🥉)- **Grouped bar chart** — Accuracy, F1, Precision, Recall- **Radar chart** — model profilleri- Detay tablo — tüm metrikler + AUC-ROC + Log Loss + süre- MLflow run geçmişi (son 15 run)### 7. 🏆 En İyi Model (`6_En_İyi_Model.py`)- Model seçici (dropdown)- 5 KPI kartı- **Confusion Matrix** — 2×2 heatmap- **ROC Curve** — operating point + random baseline- **Feature Importance** — horizontal bar (top 15)- Türetilmiş feature vurgusu

## Zorunlu Görseller — Karşılanma Durumu| # | Zorunlu Görsel | Tür | Sayfa | Durum ||---|----------------|-----|-------|-------|| 1 | 5 modelin performans karşılaştırması | Grouped bar chart | Model Karşılaştırma | ✅ || 2 | Feature Importance | Horizontal bar chart | En İyi Model | ✅ || 3 | Zaman serisi trend | Line / area chart | EDA | ✅ || 4 | Veri dağılım (histogram) | Histogram | EDA | ✅ || 5 | Veri dağılım (pie chart) | Donut chart | EDA | ✅ || 6 | EDA ek görsel 1 | Korelasyon heatmap | EDA | ✅ || 7 | EDA ek görsel 2 | Eksik değer bar | EDA | ✅ || 8 | EDA ek görsel 3 | Violin plot | Feature Eng. | ✅ || 9 | Confusion Matrix | Heatmap | En İyi Model | ✅ || 10 | ROC Curve | Line chart | En İyi Model | ✅ |**Toplam: 10 görsel ≥ 6 zorunlu** ✅

In [ ]:
# Dashboard dosya yapısını gösterimport osfrom pathlib import Pathapp_dir = Path(os.pardir) / "streamlit_app"if app_dir.exists():    print("streamlit_app/")    for item in sorted(app_dir.iterdir()):        if item.name.startswith("."):            continue        if item.is_dir():            print(f"  📁 {item.name}/")            for sub in sorted(item.iterdir()):                size = sub.stat().st_size                print(f"      {sub.name:<35} {size:>6} bytes")        else:            size = item.stat().st_size            print(f"  📄 {item.name:<35} {size:>6} bytes")else:    print("streamlit_app/ dizini bulunamadı.")

In [ ]:
# Dashboard tema ayarlarını göstertheme_path = Path(os.pardir) / "streamlit_app" / "theme.py"if theme_path.exists():    with open(theme_path) as f:        content = f.read()    print(f"theme.py — {len(content.splitlines())} satır")    print("═" * 50)    # Sadece PALETTE ve ana fonksiyon tanımlarını göster    for line in content.splitlines():        if "PALETTE" in line or "def " in line or "apply_theme" in line:            print(f"  {line.strip()}")else:    print("theme.py bulunamadı.")

## Çalıştırma```bash# Dashboard + API başlat (Docker Compose)docker compose up streamlit-dashboard dashboard-api# Veya lokal çalıştırmacd streamlit_apppip install -r requirements.txtstreamlit run app.py```## Canlı Akış Mekanizması```  Flask API (port 5001)          Streamlit (port 8501)  ┌─────────────────────┐       ┌─────────────────────┐  │ KafkaConsumer thread │──SSE──▶│ components.html()   │  │ RowCount thread     │  1s   │ EventSource JS       │  │ /api/stream         │       │ Canvas + CSS anim.   │  └─────────────────────┘       └─────────────────────┘```- **SSE (Server-Sent Events):** Her 1 saniyede Kafka + Delta Lake durumu push edilir- **Sayfa yenilemesiz:** Vanilla JS + EventSource — canlı güncelleme- **Progressive reconnect:** Bağlantı koparsa 2s → 4s → 6s → max 10s aralıkla yeniden bağlanır- **Sağlık kontrolü:** 10s event gelmezse sarı uyarı gösterilir